# Combo Strategy v2 — Per-Symbol Parameter Optimizer

**Mục tiêu:** Tối ưu **5 tham số** riêng cho từng symbol bằng `backtest_fast()` + reversal scanner.

**5 tham số được tối ưu:**

| Tham số | Ý nghĩa | Tối ưu bằng |
|---------|---------|-------------|
| **kTP** | Hệ số nhân ATR cho TP | backtest_fast |
| **x** | Breakout buffer (points) | backtest_fast |
| **trailing** | Ngưỡng kích hoạt trailing SL | backtest_fast |
| **ma_period** | Chu kỳ MA crossover | backtest_fast |
| **min_rr** | Ngưỡng R:R tối thiểu | backtest_fast |

**Khác với file 02 (Portfolio Optimal):**
- File 02: Walk-Forward + portfolio-level → tìm tổ hợp tối ưu cho **toàn bộ danh mục**
- File 03 (này): Grid search trực tiếp → tối ưu cho **1 symbol cụ thể**

**Cấu trúc notebook:**

| Cell | Nội dung |
|------|----------|
| 1-3 | Setup: Imports, config, DB + symbol config |
| 4 | Pre-cache data vào RAM |
| 5 | Baseline — params hiện tại (scanner + backtest) |
| 6 | Signal table — bảng tín hiệu baseline |
| 7 | Grid scan — backtest_fast() cho tất cả combo 5 tham số |
| 8 | Top-20 results — xếp hạng theo score |
| 9 | Interactive widget — kéo slider, xem kết quả ngay |

> **Cách dùng:** Chạy Cell 1-6 (baseline) → Cell 7-8 (grid search) → Cell 9 (tinh chỉnh thủ công)

In [ ]:
# ── Cell 1: Imports + Bootstrap ──────────────────────────────────────────────
import warnings

warnings.filterwarnings('ignore')

import itertools
import sys
import time
from datetime import datetime

import ipywidgets as widgets
from IPython.display import HTML, clear_output

# ── Bootstrap: add project root to sys.path ──────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path().resolve().parents[2]   # combo/ → strategies/ → core_python/ → project root
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from modules.chart_builder import build_reversal_chart
from modules.db_connector import get_connection, test_connection
from strategies.combo.backtest_engine import (
    add_backtest_indicators,
    backtest_fast,
    backtest_symbol,
    calc_metrics,
    detect_signals,
    load_backtest_full,
    session_mask,
)
from strategies.combo.scan_pipeline import (
    calc_reversal_stats,
    prepare_data,
    run_reversal_scan,
)
from strategies.combo.strategy_config import (
    DEFAULT_N_BARS,
    INDICATOR_COLS,
    STRATEGY,
    SYMBOLS,
    TIMEFRAME,
    get_indicator_params,
    get_symbol_params,
)
from strategies.combo.strategy_config import (
    summary as strategy_summary,
)
from strategies.shared.theme import (
    DARK,
    EQUITY_COLORS,
    NUM_FMT,
    SIGNAL,
    color_pf,
    dark_table_props,
    setup_dark_axes,
    setup_dark_figure,
    style_reversal_row,
)

print(strategy_summary())

In [ ]:
# ── Cell 2: CONFIG — Chọn symbol và phạm vi scan ─────────────────────────────
# Mục tiêu: chỉnh tại đây trước khi chạy
#   SCAN_SYMBOL : symbol muốn tối ưu (US30, HK50, J225)
#   N_BARS      : số bar H4 lịch sử để scan (nhiều hơn = thống kê tốt hơn)
#                 Gợi ý: 300-500 bar (~2-4 tháng H4) cho kết quả tin cậy

SCAN_SYMBOL = 'US30'    # Đổi sang symbol muốn tối ưu
N_BARS      = 500       # Số bar H4 gần nhất (nhiều hơn 04_scanner để có đủ mẫu thống kê)

# ── Backtest config ───────────────────────────────────────────────────────────
# Grid scan dùng backtest_fast() để tối ưu ĐẦY ĐỦ 5 tham số
# (bao gồm x và trailing mà reversal scanner không thể đánh giá)
INITIAL_BALANCE        = 100_000
SLIPPAGE_PTS           = 2
COMMISSION_USD_PER_LOT = 3.5
_COSTS = {'slippage_pts': SLIPPAGE_PTS, 'commission_per_lot': COMMISSION_USD_PER_LOT}

# Phạm vi thời gian backtest
# ⚠ KHÔNG dùng 2025 — đó là OOS của 02_wf_optimizer.ipynb
# Nếu explore kết quả 03 rồi điều chỉnh params → OOS bị contaminate
BT_FROM = '2024-01-01'   # Gần đây → phản ánh market regime hiện tại
BT_TO   = '2024-12-31'   # Kết thúc trước 2025 để giữ OOS sạch

# ── Chart options ─────────────────────────────────────────────────────────────
SHOW_REJECTED_SIGNALS = True
SHOW_MA_LINE          = True
SHOW_ENTRY_LINES      = True
ENTRY_LINE_BARS       = 8

# ── Load current params from strategy_config ─────────────────────────────────
P      = get_indicator_params()
KTP    = P['KTP']
MIN_RR = P['MIN_RR']

# Lấy params hiện tại của symbol (từ strategy_config.py)
current_params = get_symbol_params(SCAN_SYMBOL)
print(f'Symbol       : {SCAN_SYMBOL} — {SYMBOLS[SCAN_SYMBOL]["label"]}')
print(f'N Bars       : {N_BARS} (≈ {N_BARS*4/24:.0f} ngày)')
print(f'Backtest     : {BT_FROM} → {BT_TO}')
print(f'Current kTP  : {current_params["ktp"]}')
print(f'Current x    : {current_params["x"]}')
print(f'Current trail: {current_params["trailing_activation"]}')
print(f'Current MA   : {current_params["ma_period"]}')
print(f'Min R:R      : {MIN_RR}')


Symbol       : US30 — US30 (Dow Jones)
N Bars       : 500 (≈ 83 ngày)
Backtest     : 2024-01-01 → 2025-12-31
Current kTP  : 2.8
Current x    : 13.0
Current trail: 1.25
Current MA   : 25
Min R:R      : 1.25


In [ ]:
# ── Cell 3: Symbol Config + DB Connection ────────────────────────────────────
TARGET_SYMBOLS = SYMBOLS

cfg_df = pd.DataFrame(TARGET_SYMBOLS).T[['label', 'session_hours_utc', 'x']]
cfg_df.index.name = 'symbol'
display(cfg_df)

ok = test_connection()
print('✓ SQL Server connected' if ok else '✗ Connection FAILED')

In [20]:
# =============================================================================
# CELL 4 — PRE-CACHE DATA: Tải dữ liệu vào RAM 1 lần
# =============================================================================
# Tại sao cần?
#   Grid scan sẽ chạy hàng trăm backtests. Nếu mỗi lần đọc DB thì cực chậm.
#   Tải 1 lần toàn bộ data → các lần sau dùng lại trong tích tắc.
#   Lưu data CHO TẤT CẢ SYMBOLS để widget có thể đổi symbol mà không cần reload.

print('[DB] Pre-loading data for all symbols...')
_DATA_CACHE = {}

for sym_key in SYMBOLS:
    cfg_s = SYMBOLS[sym_key]
    _DATA_CACHE[sym_key] = load_backtest_full(cfg_s['symbol_id'])
    print(f'  {sym_key}: {len(_DATA_CACHE[sym_key])} bars')

print('✓ Data cached — ready for grid scan & interactive widget')

[DB] Pre-loading data for all symbols...
  US30: 5040 bars
  UK100: 5031 bars
  HK50: 6453 bars
  J225: 5052 bars
✓ Data cached — ready for grid scan & interactive widget


In [21]:
# =============================================================================
# CELL 5 — BASELINE: Chạy cả reversal scan + backtest_fast với params hiện tại
# =============================================================================
# Mục đích: xem hiệu quả của bộ tham số HIỆN TẠI — cả 2 góc nhìn:
#   1) Reversal scanner: đếm TP/SL/Reversed (trực quan, nhìn từng tín hiệu)
#   2) backtest_fast(): metrics portfolio (PF, return%, maxDD, score)
# Đây là baseline — sau đó so sánh với params khác để biết cải thiện hay xấu đi.

# ── A) Reversal scanner baseline ─────────────────────────────────────────────
P_baseline = {
    **get_indicator_params(),
    'KTP':       current_params['ktp'],
    'MIN_RR':    MIN_RR,
    'MA_PERIOD': current_params['ma_period'],
}

result     = run_reversal_scan(SCAN_SYMBOL, N_BARS, P_baseline)
df_scan    = result['df_scan']
signals_df = result['signals_df']
cfg        = result['cfg']
stats      = calc_reversal_stats(signals_df)

# ── B) Backtest_fast baseline ────────────────────────────────────────────────
raw    = _DATA_CACHE[SCAN_SYMBOL]
df_ind = add_backtest_indicators(raw, {'MA_PERIOD': current_params['ma_period']})
bl_bt  = backtest_fast(
    SCAN_SYMBOL, df_ind, cfg,
    current_params['ktp'],
    current_params['x'],
    current_params['trailing_activation'],
    BT_FROM, BT_TO,
    INITIAL_BALANCE,
    costs=_COSTS,
)

# ── Print summary ────────────────────────────────────────────────────────────
print(f'\n{"="*68}')
print(f'  BASELINE — {SCAN_SYMBOL}  ({N_BARS} bars scan | backtest {BT_FROM}→{BT_TO})')
print(f'  Params: kTP={current_params["ktp"]}  x={current_params["x"]}  '
      f'trail={current_params["trailing_activation"]}  MA={current_params["ma_period"]}')
print(f'{"="*68}')

print('\n  ── Reversal Scanner ──')
print(f'  Total signals          : {stats["n_total"]}')
print(f'  Pass R:R >= {MIN_RR}      : {stats["n_pass"]}')
if stats['n_pass'] > 0:
    print(f'  Hit TP      ✅ : {stats["n_tp"]}  ({stats["win_pct"]:.0f}%)')
    print(f'  Hit SL      ❌ : {stats["n_sl"]}')
    print(f'  Đảo chiều   🔄 : {stats["n_reversed"]}')
    if stats['n_pass'] > 0:
        print(f'  Avg R:R (passed)       : {stats["avg_rr"]:.2f}')

print('\n  ── Backtest Engine ──')
print(f'  Trades    : {bl_bt["trades"]}')
print(f'  PF        : {bl_bt["pf"]:.2f}')
print(f'  Return    : {bl_bt["ret"]:+.1f}%')
print(f'  Max DD    : {bl_bt["maxdd"]:.1f}%')
print(f'  Score     : {bl_bt["score"]:.4f}')
print('\n  → Đây là BASELINE. Cell 7 sẽ grid search 5 tham số để tìm bộ tốt hơn.')


  BASELINE — US30  (500 bars scan | backtest 2024-01-01→2025-12-31)
  Params: kTP=2.8  x=13.0  trail=1.25  MA=25

  ── Reversal Scanner ──
  Total signals          : 33
  Pass R:R >= 1.25      : 30
  Hit TP      ✅ : 7  (24%)
  Hit SL      ❌ : 14
  Đảo chiều   🔄 : 8
  Avg R:R (passed)       : 2.29

  ── Backtest Engine ──
  Trades    : 93
  PF        : 1.28
  Return    : +6.6%
  Max DD    : 4.1%
  Score     : 0.7927

  → Đây là BASELINE. Cell 7 sẽ grid search 5 tham số để tìm bộ tốt hơn.


In [ ]:
# =============================================================================
# CELL 6 — SIGNAL TABLE: Bảng chi tiết tín hiệu baseline
# =============================================================================
# Thêm cột Result (emoji) và P&L (points) để thấy rõ kết quả từng lệnh

def _add_result_pnl(df):
    """Thêm cột result (emoji) và pnl_pts vào DataFrame tín hiệu."""
    _OUTCOME_EMOJI = {'TP': '✅ TP', 'SL': '❌ SL', 'Reversed': '🔄 Đảo chiều',
                      'Open': '⏳ Open'}
    df['result'] = df['outcome'].map(_OUTCOME_EMOJI).fillna('— Rejected')

    def _calc_pnl(r):
        if r['outcome'] == 'TP':
            return round(r['tp_dist'], 1)
        elif r['outcome'] == 'SL':
            return round(-r['sl_dist'], 1)
        elif r['outcome'] == 'Reversed':
            return round(-r['sl_dist'] * 0.5, 1)   # ước tính: đóng giữa entry và SL
        return 0.0
    df['pnl_pts'] = df.apply(_calc_pnl, axis=1)
    return df


if signals_df.empty:
    print('Không có tín hiệu nào trong khoảng này.')
else:
    tbl = signals_df.copy()
    tbl = _add_result_pnl(tbl)

    display_cols = ['bar_time','direction','is_reversal','result','pnl_pts',
                    'entry','sl','tp','rr','pass_rr','atr','sl_dist','tp_dist']
    tbl = tbl[display_cols]
    tbl['bar_time'] = tbl['bar_time'].dt.strftime('%Y-%m-%d %H:%M')

    styled = (tbl.style
        .apply(style_reversal_row, axis=1)
        .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if 'TP' in str(v)
                        else (f'color:{SIGNAL["sl"]};font-weight:bold' if 'SL' in str(v)
                              else ('color:#FFB347;font-weight:bold' if 'Đảo' in str(v)
                                    else ''))),
             subset=['result'])
        .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if v > 0
                        else (f'color:{SIGNAL["sl"]};font-weight:bold' if v < 0
                              else 'color:#555')),
             subset=['pnl_pts'])
        .format({**NUM_FMT, 'pnl_pts': '{:+.1f}'})
        .set_caption(
            f'Baseline Signals — {SCAN_SYMBOL}  |  '
            f'kTP={current_params["ktp"]}  x={current_params["x"]}  '
            f'✅ TP  ❌ SL  🔄 Đảo chiều  Min R:R={MIN_RR}'
        )
    )
    display(styled)

    # ── Tóm tắt P&L ──────────────────────────────────────────────────────
    passed = tbl[tbl['pass_rr']]
    if not passed.empty:
        total_pnl = passed['pnl_pts'].sum()
        tp_pnl    = passed[passed['pnl_pts'] > 0]['pnl_pts'].sum()
        sl_pnl    = passed[passed['pnl_pts'] < 0]['pnl_pts'].sum()
        pnl_color = SIGNAL['tp'] if total_pnl >= 0 else SIGNAL['sl']
        print('\n  P&L Summary (passed signals only):')
        print(f'    ✅ TP total : +{tp_pnl:.1f} pts')
        print(f'    ❌ SL total : {sl_pnl:.1f} pts')
        print(f'    Net P&L    : {total_pnl:+.1f} pts')

,bar_time,direction,is_reversal,result,pnl_pts,entry,sl,tp,rr,pass_rr,atr,sl_dist,tp_dist
0,2025-12-08 22:00,SELL,False,❌ SL,-321.8,47598.00,47919.80,47118.24,1.49,True,171.34,321.80,479.76
1,2025-12-11 01:00,BUY,False,— Rejected,+0.0,48207.80,47676.60,48864.48,1.24,False,234.53,531.20,656.68
2,2025-12-16 09:00,SELL,False,❌ SL,-117.3,48233.20,48350.50,47758.30,4.05,True,169.61,117.30,474.90
3,2025-12-16 21:00,SELL,False,🔄 Đảo chiều,-254.8,47953.70,48463.30,47289.85,1.30,True,237.09,509.60,663.85
4,2025-12-19 21:00,BUY,True,🔄 Đảo chiều,-192.9,48310.10,47924.20,48937.78,1.63,True,224.17,385.90,627.68
5,2025-12-29 22:00,SELL,True,✅ TP,+344.1,48374.40,48573.70,48030.25,1.73,True,122.91,199.30,344.15
6,2026-01-05 06:00,BUY,False,❌ SL,-94.7,48416.90,48322.20,48928.63,5.40,True,182.76,94.70,511.73
7,2026-01-08 09:00,SELL,False,❌ SL,-179.3,48856.70,49036.00,48357.00,2.79,True,178.47,179.30,499.70
8,2026-01-12 14:00,SELL,False,❌ SL,-211.4,49108.80,49320.20,48634.42,2.24,True,169.42,211.40,474.38
9,2026-01-13 21:00,SELL,False,🔄 Đảo chiều,-216.6,49185.80,49619.00,48617.74,1.31,True,202.88,433.20,568.06



  P&L Summary (passed signals only):
    ✅ TP total : +4652.1 pts
    ❌ SL total : -6262.5 pts
    Net P&L    : -1610.4 pts


In [ ]:
# =============================================================================
# CELL 7 — GRID SCAN: Tối ưu 5 tham số bằng backtest_fast()
# =============================================================================
# Dùng backtest_fast() thay vì reversal scanner → tối ưu được TẤT CẢ tham số:
#   kTP, x, trailing_activation, ma_period, min_rr
#
# backtest_fast() mô phỏng đầy đủ:
#   - Position sizing (risk per trade)
#   - Trailing stop logic (trailing_activation)
#   - Partial TP (close 50% → trail remaining)
#   - FTMO rules (daily limit, max DD)
#   - Slippage + commission
#
# Lưới tham số — CHỈNH TẠI ĐÂY nếu muốn phạm vi khác:

GRID_KTP     = [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0]
GRID_X       = None    # None = chỉ dùng x hiện tại. Đặt list để tối ưu, vd: [5,8,10,13,15,20]
GRID_TRAIL   = [0.5, 0.75, 1.0, 1.25, 1.5]
GRID_MA      = [15, 20, 25]
GRID_MIN_RR  = [1.0, 1.25, 1.5]

# Nếu GRID_X = None, giữ x hiện tại (không tối ưu x)
if GRID_X is None:
    GRID_X = [current_params['x']]

combos = list(itertools.product(GRID_KTP, GRID_X, GRID_TRAIL, GRID_MA, GRID_MIN_RR))
print(f'Grid scan: {len(combos)} combinations')
print(f'  kTP      : {GRID_KTP}')
print(f'  x        : {GRID_X}')
print(f'  trailing : {GRID_TRAIL}')
print(f'  MA       : {GRID_MA}')
print(f'  Min R:R  : {GRID_MIN_RR}')
print(f'  Backtest : {BT_FROM} → {BT_TO}')
print()

# ── Grid search loop ──────────────────────────────────────────────────────────
grid_results = []
t0 = time.time()
cfg_sym = SYMBOLS[SCAN_SYMBOL]
raw     = _DATA_CACHE[SCAN_SYMBOL]

# Pre-compute indicators per ma_period (tránh tính lại nhiều lần)
_IND_CACHE = {}
for ma_p in GRID_MA:
    _IND_CACHE[ma_p] = add_backtest_indicators(raw, {'MA_PERIOD': ma_p})
print(f'  Indicators pre-computed for MA={GRID_MA}')

for i, (ktp, x_val, trail, ma_p, min_rr) in enumerate(combos):
    if i % 50 == 0:
        print(f'  {i}/{len(combos)} ({100*i/len(combos):.0f}%)...', end='\r')

    df_ind = _IND_CACHE[ma_p]

    # Chạy backtest_fast với bộ params này
    try:
        m = backtest_fast(
            SCAN_SYMBOL, df_ind, cfg_sym,
            ktp, x_val, trail,
            BT_FROM, BT_TO,
            INITIAL_BALANCE,
            strategy={'min_rr': min_rr},
            costs=_COSTS,
        )
    except Exception:
        continue

    # Lưu kết quả
    grid_results.append({
        'ktp':       ktp,
        'x':         x_val,
        'trailing':  trail,
        'ma_period': ma_p,
        'min_rr':    min_rr,
        'trades':    m['trades'],
        'pf':        m['pf'],
        'ret':       round(m['ret'], 2),
        'maxdd':     round(m['maxdd'], 2),
        'score':     round(m['score'], 4),
    })

elapsed = time.time() - t0
grid_df = pd.DataFrame(grid_results)

print(f'\nDone: {len(grid_results)} combos in {elapsed:.1f}s')
print(f'Combos with score > 0: {(grid_df["score"] > 0).sum()}')

Grid scan: 315 combinations
  kTP      : [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0]
  x        : [13.0]
  trailing : [0.5, 0.75, 1.0, 1.25, 1.5]
  MA       : [15, 20, 25]
  Min R:R  : [1.0, 1.25, 1.5]
  Backtest : 2024-01-01 → 2025-12-31

  Indicators pre-computed for MA=[15, 20, 25]


In [ ]:
# =============================================================================
# CELL 8 — TOP RESULTS: Xếp hạng params theo score (PF × Return / MaxDD)
# =============================================================================
# Score = PF × sqrt(max(Return, 0)) / max(MaxDD, 1)
# → params cao nhất = lãi ổn định, PF tốt, drawdown thấp
# Filter: ít nhất 5 trades (đủ mẫu thống kê)

MIN_TRADES = 5

qualified = grid_df[grid_df['trades'] >= MIN_TRADES].copy()
qualified = qualified.sort_values('score', ascending=False).reset_index(drop=True)

# ── Đánh dấu baseline ────────────────────────────────────────────────────────
baseline_mask = (
    (qualified['ktp']       == current_params['ktp']) &
    (qualified['x']         == current_params['x']) &
    (qualified['trailing']  == current_params['trailing_activation']) &
    (qualified['ma_period'] == current_params['ma_period']) &
    (qualified['min_rr']    == MIN_RR)
)
baseline_rank = qualified[baseline_mask].index[0] + 1 if baseline_mask.any() else '—'

# Baseline score từ Cell 5
baseline_score = bl_bt['score']

# ── In kết quả ─────────────────────────────────────────────────────────────────
print(f'{"="*76}')
print(f'  PER-SYMBOL OPTIMIZER — {SCAN_SYMBOL}  (backtest {BT_FROM}→{BT_TO})')
print(f'  Qualified combos: {len(qualified)} (min {MIN_TRADES} trades)')
print(f'  Baseline rank   : #{baseline_rank}  (score={baseline_score:.4f})')
print(f'{"="*76}')

# Top 20
top_n = min(20, len(qualified))
display_df = qualified.head(top_n).copy()
display_df.index = range(1, top_n + 1)
display_df.index.name = 'Rank'

# Highlight: xanh nếu score > baseline, đỏ nếu thấp hơn
def _color_score(val):
    if val > baseline_score * 1.1:   # > 10% cải thiện
        return f'color: {SIGNAL["buy"]}; font-weight: bold'
    elif val < baseline_score * 0.9: # < 10% suy giảm
        return f'color: {SIGNAL["sl"]}'
    return ''

styled = (display_df.style
    .map(_color_score, subset=['score'])
    .map(color_pf, subset=['pf'])
    .format({
        'ktp':      '{:.1f}',
        'x':        '{:.1f}',
        'trailing': '{:.2f}',
        'min_rr':   '{:.2f}',
        'pf':       '{:.2f}',
        'ret':      '{:+.1f}%',
        'maxdd':    '{:.1f}%',
        'score':    '{:.4f}',
    })
    .set_caption(
        f'Top {top_n} Params — {SCAN_SYMBOL} | '
        f'Baseline score={baseline_score:.4f} (rank #{baseline_rank}) | '
        f'🟢 Better  🔴 Worse'
    )
    .set_properties(**dark_table_props())
)
display(styled)

# ── Gợi ý best params ─────────────────────────────────────────────────────────
if len(qualified) > 0:
    best = qualified.iloc[0]
    print(f'\n  🏆 Best params for {SCAN_SYMBOL}:')
    print(f'     kTP={best["ktp"]:.1f}  x={best["x"]:.1f}  '
          f'trail={best["trailing"]:.2f}  MA={int(best["ma_period"])}  '
          f'min_rr={best["min_rr"]:.2f}')
    print(f'     → {int(best["trades"])} trades  PF={best["pf"]:.2f}  '
          f'ret={best["ret"]:+.1f}%  dd={best["maxdd"]:.1f}%  '
          f'score={best["score"]:.4f}')
    delta_s = best['score'] - baseline_score
    if delta_s > 0:
        print(f'     → Score cải thiện +{delta_s:.4f} so với baseline')
    elif delta_s < 0:
        print(f'     → Score giảm {delta_s:.4f} so với baseline (xem lại!)')
    else:
        print('     → Score bằng baseline')

In [ ]:
# =============================================================================
# CELL 9 — INTERACTIVE WIDGET: Điều chỉnh 5 tham số live
# =============================================================================
# Khi bấm "Scan & Compare":
#   1) Chạy reversal scanner → bảng tín hiệu + chart + TP/SL counts
#   2) Chạy backtest_fast()  → PF, return%, maxDD, score
#   3) So sánh với baseline
# → Bạn thấy cả 2 góc nhìn: tín hiệu cụ thể VÀ hiệu suất backtest

# ── Controls ─────────────────────────────────────────────────────────────────
w_symbol = widgets.Dropdown(
    options=list(TARGET_SYMBOLS.keys()), value=SCAN_SYMBOL,
    description='Symbol:', style={'description_width': 'auto'},
    layout=widgets.Layout(width='200px'),
)
w_nbars = widgets.BoundedIntText(
    value=N_BARS, min=30, max=1000, step=10,
    description='N Bars:', style={'description_width': 'auto'},
    layout=widgets.Layout(width='145px'),
)
w_ktp = widgets.FloatSlider(
    value=current_params['ktp'], min=0.5, max=6.0, step=0.1,
    description='kTP:', readout_format='.1f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='300px'),
)
w_x = widgets.FloatText(
    value=current_params['x'], step=1.0,
    description='x:', style={'description_width': '70px'},
    layout=widgets.Layout(width='160px'),
)
w_trail = widgets.FloatSlider(
    value=current_params['trailing_activation'], min=0.25, max=3.0, step=0.25,
    description='Trailing:', readout_format='.2f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='300px'),
)
w_minrr = widgets.FloatSlider(
    value=MIN_RR, min=0.5, max=5.0, step=0.05,
    description='Min R:R:', readout_format='.2f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='300px'),
)
w_ma = widgets.Dropdown(
    options=[10, 15, 20, 25, 30],
    value=current_params['ma_period'],
    description='MA Period:', style={'description_width': 'auto'},
    layout=widgets.Layout(width='160px'),
)
w_show_rejected = widgets.Checkbox(value=True,  description='Show Rejected')
w_show_ma       = widgets.Checkbox(value=True,  description='Show MA')
w_show_entry    = widgets.Checkbox(value=True,  description='Show Entry/SL/TP')
w_btn = widgets.Button(
    description='▶ Scan & Compare', button_style='success',
    layout=widgets.Layout(width='140px', height='34px'),
)
w_status = widgets.Label(value='')
w_out    = widgets.Output()

# ── Lưu baseline để so sánh ──────────────────────────────────────────────────
_baseline_stats  = stats.copy()
_baseline_bt     = bl_bt.copy()
_baseline_params = current_params.copy()


def _run_optimizer(_=None):
    """Callback: chạy reversal scan + backtest_fast, so sánh với baseline."""
    sym_key   = w_symbol.value
    n_bars    = w_nbars.value
    ktp_val   = w_ktp.value
    x_val     = w_x.value
    trail_val = w_trail.value
    mrr_val   = w_minrr.value
    ma_val    = w_ma.value

    w_status.value = f'⏳ Scanning {sym_key}...'

    with w_out:
        clear_output(wait=True)
        try:
            cfg_i = SYMBOLS[sym_key]

            # ── A) Reversal scanner ───────────────────────────────────────
            p_test = {
                **get_indicator_params(),
                'KTP':       ktp_val,
                'MIN_RR':    mrr_val,
                'MA_PERIOD': ma_val,
            }
            _orig_x = SYMBOLS[sym_key]['x']
            SYMBOLS[sym_key]['x'] = x_val
            try:
                res  = run_reversal_scan(sym_key, n_bars, p_test)
            finally:
                SYMBOLS[sym_key]['x'] = _orig_x

            df_sc  = res['df_scan']
            sigs   = res['signals_df']
            st     = calc_reversal_stats(sigs)

            # ── B) Backtest_fast ──────────────────────────────────────────
            raw_sym = _DATA_CACHE[sym_key]
            df_ind  = add_backtest_indicators(raw_sym, {'MA_PERIOD': ma_val})
            bt = backtest_fast(
                sym_key, df_ind, cfg_i,
                ktp_val, x_val, trail_val,
                BT_FROM, BT_TO,
                INITIAL_BALANCE,
                strategy={'min_rr': mrr_val},
                costs=_COSTS,
            )

            # ── Delta vs baseline ─────────────────────────────────────────
            bl_s = _baseline_stats
            bl_b = _baseline_bt
            d_win   = st['win_pct'] - bl_s['win_pct']
            d_score = bt['score']   - bl_b['score']
            d_pf    = bt['pf']      - bl_b['pf']
            d_ret   = bt['ret']     - bl_b['ret']

            win_color   = SIGNAL['buy'] if d_win   >= 0 else SIGNAL['sl']
            score_color = SIGNAL['buy'] if d_score >= 0 else SIGNAL['sl']
            pf_color    = SIGNAL['buy'] if d_pf    >= 0 else SIGNAL['sl']

            # ── Summary bar ───────────────────────────────────────────────
            display(HTML(f'''
<div style="margin:4px 0; padding:10px 14px; background:{DARK['panel']};
            border-left:3px solid #FFB347; font-family:monospace;
            color:{DARK['text']}; font-size:13px;">
  <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:6px;">
    <span><b>{sym_key}</b> · kTP={ktp_val:.1f} · x={x_val} ·
      trail={trail_val:.2f} · MA={ma_val} · min_rr={mrr_val:.2f}</span>
  </div>
  <div style="display:flex; gap:20px; flex-wrap:wrap;">
    <span style="color:#8b949e;">── Scanner ({n_bars} bars) ──</span>
    <span>Pass: <b>{st['n_pass']}</b></span>
    <span style="color:{SIGNAL['tp']}">TP ✅ <b>{st['n_tp']}</b></span>
    <span style="color:{SIGNAL['sl']}">SL ❌ <b>{st['n_sl']}</b></span>
    <span style="color:#FFB347">Đảo 🔄 <b>{st['n_reversed']}</b></span>
    <span style="color:{win_color}; font-weight:bold;">Win: {st['win_pct']:.0f}%</span>
  </div>
  <div style="display:flex; gap:20px; flex-wrap:wrap; margin-top:4px;">
    <span style="color:#8b949e;">── Backtest ({BT_FROM}→{BT_TO}) ──</span>
    <span>Trades: <b>{bt['trades']}</b></span>
    <span style="color:{pf_color}">PF: <b>{bt['pf']:.2f}</b></span>
    <span>Ret: <b>{bt['ret']:+.1f}%</b></span>
    <span>DD: <b>{bt['maxdd']:.1f}%</b></span>
    <span style="color:{score_color}; font-weight:bold;">Score: {bt['score']:.4f}</span>
  </div>
  <div style="margin-top:8px; padding-top:6px; border-top:1px solid #30363d;
              font-size:12px; color:#8b949e;">
    <b>vs Baseline</b> (kTP={_baseline_params['ktp']} x={_baseline_params['x']}
      trail={_baseline_params['trailing_activation']} MA={_baseline_params['ma_period']}):
    <span style="color:{win_color}"> Win {d_win:+.0f}%</span> ·
    <span style="color:{score_color}"> Score {d_score:+.4f}</span> ·
    <span style="color:{pf_color}"> PF {d_pf:+.2f}</span> ·
    <span> Ret {d_ret:+.1f}%</span>
  </div>
</div>
'''))

            # ── Signal table ──────────────────────────────────────────────
            if not sigs.empty:
                sigs = sigs.copy()
                sigs = _add_result_pnl(sigs)
                sigs['C1_Candle']   = sigs['direction'].map({'BUY': 'Bull ✓', 'SELL': 'Bear ✓'})
                sigs['C2_MA_Cross'] = sigs['direction'].map({'BUY': '↑ Cross ✓', 'SELL': '↓ Cross ✓'})
                sigs['C3_MACD']     = sigs.apply(lambda r: f'{r["macd_h"]:+.1f} ✓', axis=1)
                sigs['C4_RR']       = sigs.apply(
                    lambda r: f'{r["rr"]:.2f} ✓' if r['pass_rr'] else f'{r["rr"]:.2f} ✗', axis=1)

                dcols = ['bar_time', 'direction', 'is_reversal', 'result', 'pnl_pts',
                         'C1_Candle', 'C2_MA_Cross', 'C3_MACD', 'C4_RR',
                         'entry', 'sl', 'tp', 'atr', 'sl_dist', 'tp_dist']
                tbl = sigs[dcols].copy()
                tbl['bar_time'] = tbl['bar_time'].dt.strftime('%Y-%m-%d %H:%M')

                cond_cols = ['C1_Candle', 'C2_MA_Cross', 'C3_MACD', 'C4_RR']
                display(tbl.style
                    .apply(style_reversal_row, axis=1)
                    .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if 'TP' in str(v)
                                    else (f'color:{SIGNAL["sl"]};font-weight:bold' if 'SL' in str(v)
                                          else ('color:#FFB347;font-weight:bold' if 'Đảo' in str(v)
                                                else ''))),
                         subset=['result'])
                    .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if v > 0
                                    else (f'color:{SIGNAL["sl"]};font-weight:bold' if v < 0
                                          else 'color:#555')),
                         subset=['pnl_pts'])
                    .map(lambda v: f'color:{SIGNAL["buy"]};font-weight:bold' if '✓' in str(v)
                         else (f'color:{SIGNAL["sl"]};font-weight:bold' if '✗' in str(v)
                               else 'color:#555'),
                         subset=cond_cols)
                    .format({**NUM_FMT, 'pnl_pts': '{:+.1f}'})
                )
            else:
                print('Không có tín hiệu nào trong khoảng này.')

            # ── Chart ─────────────────────────────────────────────────────
            display(HTML(f'''
<div style="margin:22px 0 0 0; padding:7px 16px; background:#0d1117;
            border-top:1px solid #30363d; font-family:monospace;
            font-size:12px; color:#8b949e;">
  📊 <b style="color:#c9d1d9;">REVERSAL CHART</b> · {cfg_i['label']} · {n_bars} bars
  · kTP={ktp_val:.1f} · x={x_val} · trail={trail_val:.2f}
</div>
'''))

            _p = {
                **p_test,
                'ENTRY_LINE_BARS':  ENTRY_LINE_BARS,
                'SHOW_REJECTED':    w_show_rejected.value,
                'SHOW_MA':          w_show_ma.value,
                'SHOW_ENTRY_LINES': w_show_entry.value,
            }
            cfg_i_patched = {**cfg_i, 'x': x_val}
            fig = build_reversal_chart(df_sc, sigs, cfg_i_patched, sym_key, _p)
            display(fig)

            w_status.value = (
                f'✓ {sym_key} | Win={st["win_pct"]:.0f}% (Δ{d_win:+.0f}%) | '
                f'Score={bt["score"]:.4f} (Δ{d_score:+.4f})'
            )

        except Exception as e:
            import traceback
            traceback.print_exc()
            w_status.value = f'⚠ Lỗi: {e}'


def _on_symbol_change(change):
    """Khi đổi symbol → cập nhật slider values theo config hiện tại của symbol."""
    if change['name'] != 'value':
        return
    sp = get_symbol_params(change['new'])
    w_ktp.value   = sp['ktp']
    w_x.value     = sp['x']
    w_trail.value = sp['trailing_activation']
    w_ma.value    = sp['ma_period']
    _run_optimizer()


w_symbol.observe(_on_symbol_change, names='value')
w_btn.on_click(_run_optimizer)

display(widgets.VBox([
    widgets.HBox([w_symbol, w_nbars, w_ma],
                 layout=widgets.Layout(align_items='center', gap='12px')),
    widgets.HBox([widgets.VBox([w_ktp, w_trail, w_minrr]),
                  widgets.VBox([w_x]),
                  w_btn, w_status],
                 layout=widgets.Layout(align_items='center', gap='12px')),
    widgets.HBox([w_show_rejected, w_show_ma, w_show_entry]),
    w_out,
]))

# Chạy lần đầu với params hiện tại
_run_optimizer()